### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    pi.category,
    pi.sub_category,
    pi.cost_price
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "ProductInfo" pi
    ON oi.product_id = pi.product_id
WHERE o.order_status IN ('Completed', 'Shipped');
"""

df_order_completed_shipped = pd.read_sql(sql, engine)

# 查看数据
df_order_completed_shipped

### Calculate `total_price_before_tax` per customer 

In [ ]:
# Order表事先已计算好了`total_price_before_tax`了

### Calculate `AOV` per customer

In [ ]:
# 先取每张订单的收入（去重订单行）
df_revenue_per_order = (
    df_order_completed_shipped[["order_id", "customer_id", "total_price_before_tax"]]
    .drop_duplicates(subset=["order_id"])
)

# 按 customer_id 汇总总收入和订单数
df_aov_per_customer = (
    df_revenue_per_order
    .groupby("customer_id", as_index=False)
    .agg(
        revenue=("total_price_before_tax", "sum"),
        order_count=("order_id", "nunique"),
    )
)

# 计算 AOV
df_aov_per_customer["aov"] = (
    df_aov_per_customer["revenue"] / df_aov_per_customer["order_count"]
)

df_aov_per_customer

In [ ]:
# 关闭数据库连接
engine.dispose()